In [1]:
from Bio import Entrez
import pandas as pd
import time

Entrez.email = "samireformed00786@gmail.com"

# ── FUNCTIONS ──────────────────────────────────────────────────

def search_geo(query, retmax=100):
    try:
        handle = Entrez.esearch(db="gds", term=query, retmax=retmax, sort="pub+date")
        record = Entrez.read(handle)
        handle.close()
        return record['IdList']
    except Exception as e:
        print(f"    [search error: {e}]")
        return []

def fetch_summaries(id_list):
    if not id_list:
        return []
    try:
        ids = ",".join(id_list)
        handle = Entrez.esummary(db="gds", id=ids)
        records = Entrez.read(handle)
        handle.close()
        return records
    except Exception as e:
        print(f"    [fetch error: {e}]")
        return []

def to_dataframe(summaries):
    rows = []
    for s in summaries:
        rows.append({
            'accession':  s.get('Accession', ''),
            'title':      s.get('title', ''),
            'organism':   s.get('taxon', ''),
            'n_samples':  int(s.get('n_samples', 0)),
            'date':       s.get('PDAT', ''),
            'has_pub':    len(s.get('PubMedIds', [])) > 0,
            'type':       s.get('gdsType', ''),
            'platform':   s.get('GPL', ''),
            'summary':    s.get('summary', '').lower(),
        })
    return pd.DataFrame(rows)

# ── QUERIES ────────────────────────────────────────────────────
# GPL570   = Affymetrix HG-U133 Plus 2.0  (most common)
# GPL6244  = Affymetrix Human Gene 1.0 ST
# GPL10558 = Illumina HumanHT-12 v4
# GPL6947  = Illumina HumanHT-12 v3
# GPL13667 = Affymetrix HG-U219
# GPL6480  = Agilent-014850 Whole Human Genome

geo2r_queries = [

    # ── PARKINSON'S ──────────────────────────────────────────────
    "Parkinson's disease AND control AND GPL570",
    "Parkinson's disease AND control AND GPL6244",
    "Parkinson's disease AND control AND GPL10558",
    "Parkinson's disease AND control AND GPL6947",
    "Parkinson's disease AND substantia nigra AND GPL570",
    "Parkinson's disease AND blood AND GPL570",
    "Parkinson's disease AND blood AND GPL10558",
    "Parkinson's disease AND brain AND GPL6244",
    "GBA1 AND Parkinson AND GPL570",
    "LRRK2 AND Parkinson AND GPL570",
    "SNCA AND Parkinson AND GPL570",
    "alpha-synuclein AND Parkinson AND GPL570",

    # ── ALZHEIMER'S ──────────────────────────────────────────────
    "Alzheimer's disease AND control AND GPL570",
    "Alzheimer's disease AND control AND GPL6244",
    "Alzheimer's disease AND control AND GPL10558",
    "Alzheimer's disease AND hippocampus AND GPL570",
    "Alzheimer's disease AND cortex AND GPL570",
    "Alzheimer's disease AND blood AND GPL10558",

    # ── ALS ──────────────────────────────────────────────────────
    "amyotrophic lateral sclerosis AND control AND GPL570",
    "amyotrophic lateral sclerosis AND control AND GPL6244",
    "ALS AND motor neuron AND GPL570",

    # ── HUNTINGTON'S ─────────────────────────────────────────────
    "Huntington's disease AND control AND GPL570",
    "Huntington's disease AND striatum AND GPL570",

    # ── MULTIPLE SCLEROSIS ───────────────────────────────────────
    "multiple sclerosis AND control AND GPL570",
    "multiple sclerosis AND control AND GPL10558",
    "multiple sclerosis AND blood AND GPL6947",

    # ── GENERAL NEURODEGENERATION ────────────────────────────────
    "neurodegeneration AND patient AND control AND GPL570",
    "dopaminergic neuron AND disease AND GPL570",
]

# ── RUN ───────────────────────────────────────────────────────

all_results = []

for i, q in enumerate(geo2r_queries, 1):
    ids = search_geo(q, retmax=100)
    print(f"  [{i:>2}/{len(geo2r_queries)}]  {q[:60]:<60} → {len(ids)} datasets")

    if ids:
        summaries = fetch_summaries(ids[:50])
        df = to_dataframe(summaries)
        df['query'] = q
        all_results.append(df)

        checkpoint = pd.concat(all_results, ignore_index=True).drop_duplicates('accession')
        checkpoint.to_csv('geo2r_checkpoint.csv', index=False)
        print(f"    checkpoint saved — {len(checkpoint)} unique datasets so far")

    time.sleep(0.35)

# ── MASTER ────────────────────────────────────────────────────

master = pd.concat(all_results, ignore_index=True).drop_duplicates('accession')

# ── FILTER ────────────────────────────────────────────────────

control_keywords = ['control', 'healthy', 'normal', 'patient', 'case', 'disease vs', 'vs control']

def has_control_design(text):
    return any(kw in text for kw in control_keywords)

# Keep only microarray type — these have GEO2R
microarray_types = ['expression profiling by array', 'array']

filtered = master[
    (master['type'].str.lower().str.contains('array', na=False)) &
    (master['summary'].apply(has_control_design)) &
    (master['n_samples'] >= 10) &
    (master['n_samples'] <= 150) &
    (master['has_pub'] == False)
].copy()

filtered['score'] = filtered['n_samples'].apply(lambda x: min(x, 80))
filtered = filtered.sort_values('score', ascending=False)

# ── SAVE ──────────────────────────────────────────────────────

cols = ['accession', 'title', 'organism', 'n_samples', 'date', 'platform', 'type', 'query']

master[cols].to_csv('geo2r_all.csv', index=False)
filtered[cols].to_csv('geo2r_unpublished.csv', index=False)

print(f"\nTotal unique datasets found:    {len(master)}")
print(f"Microarray unpublished:         {len(filtered)}")
print(f"Saved: geo2r_all.csv")
print(f"Saved: geo2r_unpublished.csv")

# ── PREVIEW ───────────────────────────────────────────────────

print(f"\nTop 10 candidates:")
print(filtered[['accession', 'title', 'n_samples', 'date']].head(10).to_string(index=False))

  [ 1/28]  Parkinson's disease AND control AND GPL570                   → 38 datasets
    [fetch error: HTTP Error 500: Internal Server Error]
    checkpoint saved — 0 unique datasets so far
  [ 2/28]  Parkinson's disease AND control AND GPL6244                  → 1 datasets
    checkpoint saved — 1 unique datasets so far
  [ 3/28]  Parkinson's disease AND control AND GPL10558                 → 12 datasets
    checkpoint saved — 13 unique datasets so far
  [ 4/28]  Parkinson's disease AND control AND GPL6947                  → 5 datasets
    checkpoint saved — 18 unique datasets so far
  [ 5/28]  Parkinson's disease AND substantia nigra AND GPL570          → 53 datasets
    checkpoint saved — 68 unique datasets so far
  [ 6/28]  Parkinson's disease AND blood AND GPL570                     → 30 datasets
    checkpoint saved — 96 unique datasets so far
  [ 7/28]  Parkinson's disease AND blood AND GPL10558                   → 9 datasets
    checkpoint saved — 102 unique datasets so far
  